In [1]:
import os
import sys
import json
import torch
import torch.nn as nn
from torch.nn import functional as F

# Configuration des chemins vers le backend
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
BACKEND_PATH = os.path.join(PROJECT_ROOT, "backend")

if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

# Importation de l'architecture partagée NanoGPT
from app.models.nanogpt import NanoGPT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Environnement prêt ✓ | Périphérique : {device.upper()}")

Environnement prêt ✓ | Périphérique : CPU


In [2]:
# --- CELLULE 2 COMPLÈTE ET CORRIGÉE (SPRINT 4 - ARABE) ---
import os
import json
import torch

# 1. Chargement du dataset arabe nettoyé au Sprint 1
dataset_path = os.path.join(PROJECT_ROOT, "data", "dataset_ar.txt")
with open(dataset_path, 'r', encoding='utf-8') as f:
    lines = [line.strip() for line in f if line.strip()]

text = "\n".join(lines) + "\n"

# 2. Extraction du vocabulaire de caractères arabes
chars = sorted(list(set(text)))
vocab_size_ar = len(chars)

# Tables de correspondance
stoi_ar = { ch:i for i,ch in enumerate(chars) }
itos_ar = { i:ch for i,ch in enumerate(chars) }
encode_ar = lambda s: [stoi_ar[c] for c in s]
decode_ar = lambda l: ''.join([itos_ar[i] for i in l])

# MODIFICATION SÉCURISÉE POUR LE BACKEND : On encapsule stoi_ar dans l'objet global attendu {"stoi": ...}
vocab_package_ar = {"stoi": stoi_ar}

# Sauvegarde automatique dans le dossier weights du backend au format valide
weights_dir = os.path.join(BACKEND_PATH, "app", "weights")
os.makedirs(weights_dir, exist_ok=True)

with open(os.path.join(weights_dir, "vocab_ar.json"), "w", encoding="utf-8") as f:
    json.dump(vocab_package_ar, f, ensure_ascii=False, indent=4)

print(f"Vocabulaire Arabe sauvegardé ({vocab_size_ar} caractères uniques) au format backend ✓")

# =====================================================================
# AJOUT DES COMPOSANTES DE PRÉPARATION DE DONNÉES EN TENSEURS :
# =====================================================================

# 3. Conversion de tout le texte arabe en nombres (tenseurs PyTorch)
data_encoded = torch.tensor(encode_ar(text), dtype=torch.long)

# 4. Séparation 90% pour l'entraînement et 10% pour la validation
n = int(0.9 * len(data_encoded))
train_data = data_encoded[:n]
val_data = data_encoded[n:]

print(f"Données arabes prêtes : train_data ({len(train_data)} tokens) et val_data ({len(val_data)} tokens) créées ! ✓")

Vocabulaire Arabe sauvegardé (39 caractères uniques) au format backend ✓
Données arabes prêtes : train_data (10818 tokens) et val_data (1203 tokens) créées ! ✓


In [3]:
# Définition des hyperparamètres
batch_size = 32
block_size = 24  # Plus grand pour absorber les préfixes de conditionnement

# Exemple de tokenisation conditionnelle simple : 
# On peut ajouter des caractères spéciaux comme 'M' pour Marque et 'S' pour Société au début de la séquence
data_encoded = torch.tensor(encode_ar(text), dtype=torch.long)

def get_batch_ar(split):
    # Séparation train/val basique
    n = int(0.9 * len(data_encoded))
    data = data_encoded[:n] if split == 'train' else data_encoded[n:]
    
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch_ar('train')
print(f"Forme du batch Arabe conditionnel X : {xb.shape}")

Forme du batch Arabe conditionnel X : torch.Size([32, 24])


In [4]:
model_ar = NanoGPT(
    vocab_size=vocab_size_ar, 
    n_embed=64, 
    n_head=4, 
    n_layer=4, 
    block_size=block_size
)
model_ar = model_ar.to(device)
optimizer = torch.optim.AdamW(model_ar.parameters(), lr=1e-3)

print("Modèle final nanoGPT Arabe initialisé ✓")

Modèle final nanoGPT Arabe initialisé ✓


In [5]:
max_iters = 1500  # On augmente un peu pour stabiliser l'apprentissage de l'arabe
eval_interval = 300

print("Entraînement du Transformer Arabe en cours...")

for iteration in range(max_iters):
    if iteration % eval_interval == 0 or iteration == max_iters - 1:
        model_ar.eval()
        with torch.no_grad():
            x_t, y_t = get_batch_ar('train')
            x_v, y_v = get_batch_ar('val')
            _, loss_train = model_ar(x_t, y_t)
            _, loss_val = model_ar(x_v, y_v)
        print(f"Itération {iteration:4d} | Train Loss: {loss_train.item():.4f} | Val Loss: {loss_val.item():.4f}")
        model_ar.train()

    xb, yb = get_batch_ar('train')
    logits, loss = model_ar(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\nEntraînement du modèle arabe terminé avec succès ! ✓")

Entraînement du Transformer Arabe en cours...
Itération    0 | Train Loss: 3.6698 | Val Loss: 3.6823
Itération  300 | Train Loss: 2.3362 | Val Loss: 2.8373
Itération  600 | Train Loss: 1.9462 | Val Loss: 2.7196
Itération  900 | Train Loss: 1.7647 | Val Loss: 2.4333
Itération 1200 | Train Loss: 1.5894 | Val Loss: 2.6706
Itération 1499 | Train Loss: 1.5109 | Val Loss: 2.5877

Entraînement du modèle arabe terminé avec succès ! ✓


In [6]:
# 1. Exportation des poids vers le Backend
model_path = os.path.join(weights_dir, "model_ar.pt")
torch.save(model_ar.to('cpu').state_dict(), model_path)
print(f"Poids finaux arabes exportés vers : {model_path} ✓")

# 2. Test d'inférence rapide pour valider la génération de caractères arabes
model_ar.to(device)
model_ar.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)

print("\n--- Test d'Inférence : Noms arabes générés ---")
generated_tokens = model_ar.generate(context, max_new_tokens=50)[0].tolist()
print(decode_ar(generated_tokens))
print("\nSprint 4 validé ! L'application dispose désormais de tous ses fichiers de poids.")

Poids finaux arabes exportés vers : c:\Users\Lenovo\Documents\ProjetVersion1\backend\app\weights\model_ar.pt ✓

--- Test d'Inférence : Noms arabes générés ---

الغنورات

Sprint 4 validé ! L'application dispose désormais de tous ses fichiers de poids.
